<a href="https://colab.research.google.com/github/Kuper994/Software-Project/blob/main/Databases.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Notebook's installations:

In [ ]:
# installations for the lesson:

!apt-get -qq update
!apt-get -qq install -y postgresql postgresql-contrib
!service postgresql start

!sudo -u postgres psql -c "CREATE USER student WITH PASSWORD '1234';"
!sudo -u postgres psql -c "CREATE DATABASE university OWNER student;"
%pip install -q "psycopg[binary]"

!wget -qO - https://www.mongodb.org/static/pgp/server-7.0.asc | \
    gpg --dearmor -o /usr/share/keyrings/mongodb-server-7.0.gpg

!echo "deb [ arch=amd64,arm64 signed-by=/usr/share/keyrings/mongodb-server-7.0.gpg ] \
    https://repo.mongodb.org/apt/ubuntu jammy/mongodb-org/7.0 multiverse" \
    > /etc/apt/sources.list.d/mongodb-org-7.0.list

!apt-get update -qq
!apt-get install -y mongodb-org

!mkdir -p /data/db
!mongod --fork --logpath /var/log/mongodb.log --dbpath /data/db
!pip install -q pymongo

import os
from pymongo import MongoClient

os.environ["MONGODB_URI"] = "mongodb://localhost:27017/"
client = MongoClient("mongodb://localhost:27017/")
client.admin.command("ping")

print("Connected to MongoDB!")

# Databases with Python


In this lecture, we will use Python to work with data in relational and document databases, then connect both approaches to the **MapReduce** programming model.

**Learning goals**
- Explain what databases provide beyond ordinary files.
- Read and write SQL queries involving filtering, joins, and aggregation.
- Demonstrate Python applications connetcion to PostgreSQL.
- Represent and query data as NoSQL documents.
- Explain MapReduce and implement it in Python.
- Choose an approach based on the task and data model.

> If a program already has a CSV file, what would make a database worth using instead?

## 1. Why databases?

A database is managed data; a **database management system (DBMS)** provides ways to store, query, update, protect, and share that data.

A CSV file can be enough for a small, single-user task. Databases become useful when data is large, frequently updated, shared by multiple users, or needs reliable constraints and transactions.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image, Markdown

In [ ]:
# A flowchart introducing storage choices.
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

fig, ax = plt.subplots(figsize=(11, 6))
ax.set_xlim(0, 12)
ax.set_ylim(0, 7)
ax.axis("off")

def flow_box(x, y, w, h, text, facecolor="#e8f1fb", fontsize=10):
    patch = FancyBboxPatch(
        (x, y), w, h, boxstyle="round,pad=0.12,rounding_size=0.16",
        linewidth=1.5, edgecolor="#334155", facecolor=facecolor
    )
    ax.add_patch(patch)
    ax.text(x + w / 2, y + h / 2, text, ha="center", va="center",
            fontsize=fontsize, wrap=True, color="#1e293b")

def arrow(start, end, label=None, label_offset=(0, 0)):
    ax.add_patch(FancyArrowPatch(
        start, end, arrowstyle="-|>", mutation_scale=16,
        linewidth=1.6, color="#64748b", shrinkA=2, shrinkB=3
    ))
    if label:
        mx = (start[0] + end[0]) / 2 + label_offset[0]
        my = (start[1] + end[1]) / 2 + label_offset[1]
        ax.text(mx, my, label, fontsize=9, ha="center", va="center",
                color="#334155",
                bbox=dict(boxstyle="round,pad=0.25", facecolor="white",
                          edgecolor="#cbd5e1", linewidth=0.7))

flow_box(3.8, 5.7, 4.4, 1.0, "How should the application store its data?",
         "#dbeafe", 12)

flow_box(0.8, 3.2, 3.5, 1.0, "A CSV or JSON file\nmay be sufficient",
         "#e2e8f0", 10)

flow_box(6.3, 3.2, 4.9, 1.0,
         "How is the data best organized?",
         "#fef3c7", 11)

arrow((4.6, 5.7), (2.55, 4.2),
      "Small, simple application\nwith limited data sharing", (-0.5, 0.15))
arrow((7.4, 5.7), (8.75, 4.2),
      "Multiple users, concurrent updates,\nor reliable data management", (0.8, 0.15))



flow_box(3.8, 0.3, 3.5, 0.8, "SQL: relational tables", "#dbeafe", 10)
flow_box(8.15, 0.3, 3.5, 0.8, "NoSQL: other data models", "#dcfce7", 10)

arrow((7.6, 3.1), (5.3, 1.18),
      "In tables with relationships", (-0.3, 0.2))
arrow((9.9, 3.1), (9.9, 1.15),
      "In flexible, nested documents", (0.5, 0.1))

ax.set_title("Choosing a Storage Approach", fontsize=15, pad=15, weight="bold")
plt.tight_layout()
plt.show()

**Class question 1**

A teaching assistant keeps grades in a CSV file. Five TAs need to update grades at the same time, and every grade must refer to an existing student.

1. What can go wrong with the file-based approach?
2. Which database features would help?
3. Would a database automatically solve every problem?

## 2. Relational databases and SQL

In a relational database, data is organized into **tables**. Rows represent records, columns represent attributes, and keys represent identity and relationships.

We'll use a small university dataset: students enroll in courses, and each enrollment has a grade.

### Start with a file: a list of students

At first, the university only needs to save a few student records. A CSV file is a reasonable starting point: it is easy to create, read, and share.

We will begin with a file-like table, then add requirements one at a time. The goal is not to claim that files cannot do these things, but to see what a database gives us automatically.

In [ ]:
# Stage 1: represent a small CSV file as a table.
students_df = pd.DataFrame([
    {"student_id": 1, "name": "Dana", "department": "CS"},
    {"student_id": 2, "name": "Amir", "department": "Biology"},
    {"student_id": 3, "name": "Noa", "department": "CS"},
    {"student_id": 4, "name": "Lior", "department": "Physics"},
])
students_df.to_csv("students.csv", index=False)
display(students_df)
print("Saved students.csv")

**Class question:** If we only need to save and later read this small list, why might a CSV file be enough?

Now the university's needs grow:
- Students can enroll in many courses, and each course has many students.
- Every enrollment must refer to a real student and a real course.
- A student should not be enrolled in the same course twice.
- Grades must be valid, and we want to query students, courses, and grades together.

We need a way to represent relationships and protect the integrity of the data.

### What are tables, rows, and columns?

A **table** stores records of one kind. A **row** is one record; a **column** is one attribute shared by the records in that table. For example, `students` has one row per student and columns for the student's ID, name, and department.

A **relational database** stores data in tables (formally, relations) and defines how records in different tables are connected. Here, the word *relation* can mean the mathematical table in the relational model; the *relationship* between students and courses is a separate idea.

In [ ]:
# A small table: each row is a student; each column is an attribute.
display(students_df)

### What is SQL?

**SQL** (Structured Query Language) is the language used to define, query, and modify data in relational databases. SQL is *declarative*: we describe **what** result we want, rather than writing the steps to find it.

For example:

```sql
SELECT name, department
FROM students
WHERE department = 'CS';
```

Read it as: choose the `name` and `department` columns from `students`, keeping only rows whose department is `CS`.

SQL is the language; **SQLite** is a database engine that runs SQL. We use SQLite here because it runs directly in Colab without a server. The same core SQL ideas apply to PostgreSQL.

In [ ]:
# First SQL example: create one table, insert rows, and query it.
import sqlite3

conn = sqlite3.connect(":memory:")
conn.execute("PRAGMA foreign_keys = ON")
cur = conn.cursor()
cur.execute("CREATE TABLE students (student_id INTEGER, name TEXT NOT NULL, department TEXT NOT NULL)")
cur.executemany(
    "INSERT INTO students VALUES (?, ?, ?)",
    students_df[["student_id", "name", "department"]].itertuples(index=False, name=None)
)
conn.commit()

# Class poll - What would be the return value of this query?
pd.read_sql_query("SELECT name, department FROM students WHERE department = 'CS'", conn)

### How does a table identify records?

<div align="center">

**A table needs a way to uniquely identify each record.**

↓

**Primary Key (PK)**

A column (or combination of columns) that uniquely identifies each row.

- Must be **UNIQUE**
- Cannot be **NULL**

Example: `student_id` in the `students` table

↓

**Other constraints that protect data quality**

| Constraint | What does it ensure? |
|---|---|
| `NOT NULL` | A value must be provided |
| `UNIQUE` | No duplicate values |
| `CHECK` | Values satisfy a condition (e.g., `credits > 0`) |

↓

**Why does it matter?**

Two students can have the same name, but each has a different `student_id`.

**An ID provides a stable identity for every record.**

</div>

### How do tables connect?




<div align="center">

**Students and courses have a many-to-many relationship.**

One student can take many courses, and one course can have many students.

↓

**How do we represent this relationship?**

Introduce a third table: `enrollments`

Each row represents one student-course pair.

↓

**Foreign Keys (FK)**

Columns that reference keys in another table.

- `enrollments.student_id` → `students.student_id`
- `enrollments.course_id` → `courses.course_id`

↓

**Composite Primary Key (PK)**

The combination of `(student_id, course_id)` uniquely identifies each enrollment.

- Prevents duplicate student-course pairs.
- Ensures each student can enroll in a course only once.

↓

**The result: three connected tables**


| Table | Role |
|---|---|
| `students` | Stores student information |
| `courses` | Stores course information |
| `enrollments` | Connects students and courses |

</div>



### Create the three-table schema

Now that the vocabulary is in place, we can express the design in SQL. Notice that the database definition itself contains the integrity rules: primary keys, foreign keys, required fields, and checks.

In [ ]:
# Recreate the database with the complete schema.
conn = sqlite3.connect(":memory:")
conn.execute("PRAGMA foreign_keys = ON")
cur = conn.cursor()

cur.executescript('''
CREATE TABLE students (
    student_id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    department TEXT NOT NULL
);
CREATE TABLE courses (
    course_id INTEGER PRIMARY KEY,
    title TEXT NOT NULL,
    credits INTEGER NOT NULL CHECK (credits > 0)
);
CREATE TABLE enrollments (
    student_id INTEGER NOT NULL REFERENCES students(student_id),
    course_id INTEGER NOT NULL REFERENCES courses(course_id),
    grade REAL CHECK (grade BETWEEN 0 AND 100),
    PRIMARY KEY (student_id, course_id)
);
''')

cur.executemany("INSERT INTO students VALUES (?, ?, ?)", [
    (1, "Dana", "CS"), (2, "Amir", "Biology"),
    (3, "Noa", "CS"), (4, "Lior", "Physics")
])
cur.executemany("INSERT INTO courses VALUES (?, ?, ?)", [
    (101, "Algorithms", 4), (102, "Databases", 3), (103, "Genetics", 3)
])
cur.executemany("INSERT INTO enrollments VALUES (?, ?, ?)", [
    (1, 101, 95), (1, 102, 91), (2, 103, 88),
    (3, 101, 84), (3, 102, 97)
])
conn.commit()
print("University tables created and populated.")

In [ ]:
# Inspect the actual contents of all three SQL tables.
for table in ["students", "courses", "enrollments"]:
    display(Markdown(f"#### `{table}`"))
    display(pd.read_sql_query(f"SELECT * FROM {table}", conn))

**Data-integrity check:** try inserting an enrollment for a student ID that does not exist. The foreign key should reject it. The same applies to a course ID that does not exist.

In [ ]:
# This should fail because student_id=999 is not in students.
try:
    cur.execute("INSERT INTO enrollments VALUES (?, ?, ?)", (999, 101, 90))
except sqlite3.IntegrityError as error:
    print(type(error).__name__ + ":", error)

In [ ]:
# Class Poll - Why is the INSERT statement failing?
try:
    cur.execute("INSERT INTO enrollments VALUES (?, ?, ?)", (1, 101, 90))
except sqlite3.IntegrityError as error:
    print(type(error).__name__ + ":", error)

### From a well-structured database to useful questions

So far, we have designed tables that store data consistently: primary keys identify records, foreign keys connect
related records, and constraints prevent invalid data.

But storing data is only half the job. The university also needs answers to questions such as:

- Which students belong to the CS department?
- Which courses has each student taken?
- What is the average grade in each course?

**SQL queries let us ask these questions.** We will start by retrieving information from one table, then combine
related tables with `JOIN`, and finally summarize records with aggregation.


### Ask the database a question: query one table

Now that the tables are in place, we can retrieve exactly the information we need. SQL statements are composed
from clauses:

- `SELECT` chooses columns.
- `FROM` names the table.
- `WHERE` filters rows.
- `ORDER BY` sorts the result.

Use a parameter for values supplied by Python.


In [ ]:
query = '''
SELECT name, department
FROM students
WHERE department = ?
ORDER BY name;
'''
pd.read_sql_query(query, conn, params=("CS",))

**Class question:** How will the rows be sorted if we remove `ORDER BY name`

### Query across related tables: `JOIN`

A `JOIN` combines rows from tables using a matching condition. Here, the enrollment row contains IDs, while the names and course titles live in their own tables. We join on the keys to retrieve a readable report.

In [ ]:
query = '''
SELECT s.name, c.title, e.grade
FROM enrollments AS e
JOIN students AS s ON s.student_id = e.student_id
JOIN courses AS c ON c.course_id = e.course_id
WHERE e.grade >= ?
ORDER BY e.grade DESC;
'''
pd.read_sql_query(query, conn, params=(90,))

### Aggregation: summarize grades by course

SQL can summarize many rows with aggregate functions such as `COUNT` and `AVG`. `GROUP BY` defines which records belong to each group.

In [ ]:
avg_query = '''
SELECT c.title, COUNT(*) AS enrollment_count, ROUND(AVG(e.grade), 1) AS average_grade
FROM enrollments AS e
JOIN courses AS c ON c.course_id = e.course_id
GROUP BY c.course_id, c.title
ORDER BY average_grade DESC;
'''
pd.read_sql_query(avg_query, conn)

### Why PostgreSQL?

SQLite is convenient for a notebook demonstration because it needs no server. In a real university application, multiple services and staff may need to connect to a shared database server. PostgreSQL is one option for that setup.

A Python application connects through a driver such as `psycopg`.

In [ ]:
import os
import psycopg

# Use environment variables if configured; otherwise use the local Colab server.
PG_CONFIG = {
    "host": os.getenv("PGHOST", "localhost"),
    "port": os.getenv("PGPORT", "5432"),
    "dbname": os.getenv("PGDATABASE", "university"),
    "user": os.getenv("PGUSER", "student"),
    "password": os.getenv("PGPASSWORD", "1234")
}

try:
    with psycopg.connect(**PG_CONFIG) as conn:
        with conn.cursor() as cur:
            cur.execute("SELECT version();")
            version = cur.fetchone()[0]

    print("Connected successfully to PostgreSQL!")
    print(version)

except psycopg.OperationalError as e:
    print("Could not connect to PostgreSQL.")
    print("Check that the server is running and the credentials are correct.")
    print(e)

A PostgreSQL query uses the same SQL ideas, but `psycopg` uses `%s` placeholders for values:

```python
with pg_conn.cursor() as cur:
    cur.execute(
        "SELECT student_id, name FROM students WHERE department = %s",
        ("CS",)
    )
    rows = cur.fetchall()
```

**Class question:** Why is it risky to build a query by concatenating a string supplied by a user?

## 3. A new requirement: degree programs have different structures

So far, our university database has stored students, courses, and enrollments. These records fit naturally
into relational tables: each kind of record has a clear set of attributes, and foreign keys connect related
records.

Now imagine that the university wants to store information about its **degree programs**. The university
offers Bachelor's, Master's, and PhD programs, but the information required for each level is not the same:

- **Bachelor's:** coursework credits, required courses, electives, and a final project.
- **Master's:** coursework credits, a supervisor, and a thesis or non-thesis track.
- **PhD:** a supervisor, research topic, thesis, publication requirements, and research milestones.

This new requirement gives us a reason to consider whether every record needs to follow exactly the same structure.


Each degree level has its own structure and degree-specific information:

| Requirement | BSc Computer Science | MSc Computer Science | PhD |
|---|---|---|---|
| Total coursework credits | 120 | 36 | 12 |
| Required courses | Algorithms, Databases, Operating Systems | Research Methods | — |
| Elective credits | 24 | — | — |
| Final project | Required (6 credits) | — | — |
| Supervisor | — | Dr. Colonel Mustard | Prof. Plum |
| Study track | — | Thesis | — |
| Thesis topic | — | Graph Neural Networks for Social Networks | Efficient Algorithms for Large-Scale Graph Processing |
| Thesis credits | — | 12 | — |
| Minimum publications | — | — | 2 |
| Research milestones | — | — | Research proposal, Progress review, Thesis submission |

**Discussion:** How could we represent these programs in our relational database? Would every program need
the same columns? What would the empty values mean?


### One requirement, different possible database designs

A relational database can represent these requirements. For example, we could create separate tables for
degree programs, course requirements, supervisors, theses, and publications, and connect them with keys.
Another option is to use one wide table, although many columns would be empty for most degree types.

These are valid relational designs, but they require us to decide how to represent each type of program
using tables and relationships.

There is another way to organize this information: store each degree program as a single **document**.
A document can contain nested objects and lists, and different documents can contain different fields.
This is the idea behind a **document database**.


In [ ]:
# Represent each degree program as a document (a Python dictionary).
# Notice that each document contains the fields relevant to that degree level.
degree_programs = [
    {
        "degree": "PhD Computer Science",
        "level": "PhD",
        "supervisor": "Prof. Plum",
        "coursework": {"total_credits": 12},
        "research": {
            "thesis_topic": "Efficient Algorithms for Large-Scale Graph Processing",
            "minimum_publications": 2,
            "milestones": [
                "Research proposal",
                "Progress review",
                "Thesis submission"
            ]
        }
    },
    {
        "degree": "MSc Computer Science",
        "level": "Master",
        "supervisor": "Dr. Colonel Mustard",
        "coursework": {
            "total_credits": 36,
            "required_courses": ["Research Methods"]
        },
        "track": {
            "type": "thesis",
            "thesis_topic": "Graph Neural Networks for Social Networks",
            "thesis_credits": 12
        }
    },
    {
        "degree": "BSc Computer Science",
        "level": "Bachelor",
        "coursework": {
            "total_credits": 120,
            "required_courses": ["Algorithms", "Databases", "Operating Systems"],
            "elective_credits": 24
        },
        "final_project": {"required": True, "credits": 6}
    },
]

display(pd.json_normalize(degree_programs).fillna("—"))


### From Python dictionaries to a document database

The dictionaries above demonstrate the **document model** in Python. Each dictionary represents one
degree program, with nested dictionaries and lists for information such as coursework and research.
The documents share a few fields (such as `degree` and `level`), but they do not need to have identical
structures.

A document database stores this kind of data directly. Its main concepts are:

| Relational database | Document database |
|---|---|
| Table | Collection |
| Row | Document |
| Column | Field |
| Fixed table schema | Flexible document structure |

For example, the three dictionaries in `degree_programs` could be stored as three documents in a
collection named `degree_programs`.


### What does this structure give us?

- **Flexible structure:** documents can have different fields when different degree levels require different information.
- **Nested data:** related details, such as coursework or research milestones, can be kept together in one document.
- **Validation is still important:** the application or database should check required fields and data types.
- **Relationships still exist:** if documents refer to other entities, we must decide how to represent and maintain those references.

This flexibility is useful when records naturally have different structures. It does not mean that a document
database is always better than a relational database; the choice depends on the data and the questions we need
to answer.



The examples below use **MongoDB**, a popular document database,
to show how we can store and query them.

**Important:** MongoDB is one implementation of the document-database approach; the document model itself
is not specific to MongoDB.

### First, query the documents in Python

Before connecting to a database, let's see what it means to query this document structure.
Because `degree_programs` is a Python list of dictionaries, we can filter it with a list comprehension.

For example, select the PhD program and display its supervisor and research details. In the next step,
we will write the equivalent query in MongoDB.


In [ ]:
phd_programs = [
    {
        "degree": degree["degree"],
        "supervisor": degree["supervisor"],
        **degree["research"]
    }
    for degree in degree_programs
    if degree["level"] == "PhD"
]
pd.DataFrame(phd_programs)


### The same query in MongoDB

The Python example filtered a list of dictionaries. MongoDB provides database operations for doing the
same thing on documents stored in a collection.

In MongoDB, a **collection** is a group of documents. The `find()` operation selects documents that match
a filter. Since `minimum_publications` is nested inside the `research` object, we refer to it using
dot notation:

```python
list(degree_collection.find(
    {"level": "PhD", "research.minimum_publications": {"$exists": True}},
    {"_id": 0, "degree": 1, "supervisor": 1, "research": 1}
))
```

The filter asks for PhD documents that contain `research.minimum_publications`. The second dictionary
specifies which fields to return: `1` includes a field and `_id: 0` excludes MongoDB's automatically
generated document ID.

To connect to a real MongoDB server from Python, we use the `pymongo` module. The following cell would run only after running the first cell of this notebook.


In [ ]:

MONGODB_URI = os.getenv("MONGODB_URI")

if MONGODB_URI:
    from pymongo import MongoClient
    client = MongoClient(MONGODB_URI, serverSelectionTimeoutMS=5000)
    client.admin.command("ping")
    degree_collection = client["university_demo"]["degree_programs"]

    # Insert the documents from the Python example.
    degree_collection.delete_many({"degree": {"$in": [d["degree"] for d in degree_programs]}})
    degree_collection.insert_many(degree_programs)

    # Run the same query shown above.
    print(list(degree_collection.find(
        {"level": "PhD", "research.minimum_publications": {"$exists": True}},
        {"_id": 0, "degree": 1, "supervisor": 1, "research": 1}
    )))
else:
    print("MongoDB demo skipped: configure MONGODB_URI to connect to a server.")


### Aggregating documents in MongoDB

So far, we have used MongoDB to select individual documents. We can also ask it to summarize information
across many documents.

For example, calculate the total coursework credits required across all degree programs, grouped by
degree level. MongoDB uses an **aggregation pipeline**: each stage transforms or summarizes the documents
passed to it.

```python
pipeline = [
    {"$group": {
        "_id": "$level",
        "total_coursework_credits": {"$sum": "$coursework.total_credits"}
    }},
    {"$sort": {"_id": 1}}
]
results = list(degree_collection.aggregate(pipeline))
```

`$group` creates one result per degree level and sums the nested `coursework.total_credits` field.
`$sort` orders the resulting groups by level.


## 4. Scaling Up: From Queries to MapReduce

As the university grows, we may want to answer questions across thousands of degree programs:

- How many programs per degree level?
- What is the total number of credits per level?
- How many PhD programs require publications?

**MapReduce** provides a way to process large collections of records in two stages:

- **Map:** Transform records into key-value pairs.
- **Reduce:** Aggregate values by key.



In [ ]:
# @title
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

fig, ax = plt.subplots(figsize=(15, 6))
ax.set_xlim(0, 15)
ax.set_ylim(0, 6)
ax.axis("off")

# Colors
C = {
    "input": "#E8EEF7",
    "map": "#FFF1D6",
    "group": "#FCE7E7",
    "reduce": "#E1F2E8",
    "text": "#263445",
    "arrow": "#8291A5",
}

def box(x, y, w, h, title, color):
    patch = FancyBboxPatch(
        (x, y), w, h,
        boxstyle="round,pad=0.12,rounding_size=0.15",
        facecolor=color, edgecolor="none"
    )
    ax.add_patch(patch)
    ax.text(x + w/2, y + h - 0.35, title, ha="center", va="center",
            fontsize=13, weight="bold", color=C["text"])

def arrow(x1, x2):
    ax.add_patch(FancyArrowPatch(
        (x1, 3), (x2, 3), arrowstyle="-|>",
        mutation_scale=18, linewidth=1.8, color=C["arrow"]
    ))

# Title
ax.text(7.5, 5.55, "MapReduce: Count Students by Degree Level",
        ha="center", fontsize=17, weight="bold", color=C["text"])

# Input
box(0.2, 1.0, 2.5, 3.8, "INPUT", C["input"])
for i, (name, degree) in enumerate([
    ("Alice", "PhD"),
    ("Bob", "Master"),
    ("Charlie", "PhD")
]):
    y = 3.65 - i * 0.85
    ax.text(1.45, y, f"{name}  |  {degree}", ha="center", fontsize=11,
            color=C["text"])

# Map
box(3.4, 1.0, 2.5, 3.8, "MAP", C["map"])
for i, item in enumerate(["(PhD, 1)", "(Master, 1)", "(PhD, 1)"]):
    ax.text(4.65, 3.65 - i * 0.85, item, ha="center", fontsize=11,
            family="monospace", color=C["text"])

# Group
box(6.6, 1.0, 3.0, 3.8, "GROUP", C["group"])
ax.text(8.1, 3.35, "PhD", ha="center", fontsize=11, weight="bold",
        color=C["text"])
ax.text(8.1, 2.95, "[1, 1]", ha="center", fontsize=12, family="monospace",
        color=C["text"])
ax.text(8.1, 2.1, "Master", ha="center", fontsize=11, weight="bold",
        color=C["text"])
ax.text(8.1, 1.7, "[1]", ha="center", fontsize=12, family="monospace",
        color=C["text"])

# Reduce
box(10.3, 1.0, 2.5, 3.8, "REDUCE", C["reduce"])
ax.text(11.55, 3.35, "PhD", ha="center", fontsize=11, weight="bold",
        color=C["text"])
ax.text(11.55, 2.95, "2 students", ha="center", fontsize=12,
        color=C["text"])
ax.text(11.55, 2.1, "Master", ha="center", fontsize=11, weight="bold",
        color=C["text"])
ax.text(11.55, 1.7, "1 student", ha="center", fontsize=12,
        color=C["text"])

# Arrows
for x1, x2 in [(2.8, 3.3), (6.0, 6.5), (9.7, 10.2)]:
    arrow(x1, x2)

# Bottom explanation
ax.text(7.5, 0.35,
        "Map: create (degree, 1) pairs     |     Group: collect values by degree     |     Reduce: sum the values",
        ha="center", fontsize=10, color="#64748b")

plt.tight_layout()
plt.show()

### MapReduce in Python

1. **Map:** convert each degree program into `(level, total_coursework_credits)`.
2. **Group:** collect all credit values with the same degree level.
3. **Reduce:** sum the values for each degree level.

A distributed MapReduce system can split the input across machines and combine partial results. Here, we
implement the same logical stages on a small in-memory dataset.


In [ ]:
# MAP: each degree program contributes its coursework credits to its degree level.
mapped = [
    (degree["level"], degree["coursework"]["total_credits"])
    for degree in degree_programs
]
mapped


In [ ]:
# GROUP: collect the mapped credit values by degree level.
from collections import defaultdict

grouped = defaultdict(list)
for key, value in mapped:
    grouped[key].append(value)

dict(grouped)


In [ ]:
# REDUCE: sum the coursework credits for each degree level.
credit_totals = {level: sum(values) for level, values in grouped.items()}
credit_totals


**Class question:** If we wanted to calculate the number of programs at each degree level instead of
the total coursework credits, what should the mapper emit? What should the reducer calculate?


## 5. Joint exercise:

<img src="https://drive.google.com/thumbnail?id=1DvZPW-mjivNtT0eG6fTFuUwyGZaalx-E&sz=w500" width="500">

In [ ]:
for table in ["students", "courses", "enrollments"]:
    display(Markdown(f"#### `{table}`"))
    display(pd.read_sql_query(f"SELECT * FROM {table}", conn))

query = '''
'''

# pd.read_sql_query(query, conn)


Choose the **one correct answer** for each question. Some questions use features we have not covered in detail;
those are marked **Challenge**. Read the whole query before deciding what it returns.

Use the tables and documents created earlier in this notebook.

### Question 1 - JOIN, filtering, and sorting

```sql
SELECT s.name, c.title, e.grade
FROM enrollments AS e
JOIN students AS s ON s.student_id = e.student_id
JOIN courses AS c ON c.course_id = e.course_id
WHERE e.grade >= 90
ORDER BY e.grade DESC;
```

What rows are returned, in order?

A. `Dana | Algorithms | 95`, `Dana | Databases | 91`, `Noa | Databases | 97`

B. `Dana | Algorithms | 95`, `Dana | Databases | 91`, `Noa | Databases | 97`

C. `Noa | Databases | 97`, `Dana | Algorithms | 95`, `Dana | Databases | 91`, `Amir | Genetics | 88`

D. `Noa | Databases | 97`, `Dana | Algorithms | 95`, `Dana | Databases | 91`

### Question 2 — GROUP BY with multiple aggregate functions

```sql
SELECT c.title,
       COUNT(*) AS n,
       ROUND(AVG(e.grade), 1) AS avg_grade
FROM enrollments AS e
JOIN courses AS c ON c.course_id = e.course_id
GROUP BY c.course_id, c.title
ORDER BY avg_grade DESC;
```

What is the result?

A. `Databases | 2 | 94.0`, `Algorithms | 2 | 89.5`, `Genetics | 1 | 88.0`

B. `Databases | 2 | 94.0`, `Algorithms | 2 | 89.5`, `Genetics | 1 | 88.0`, sorted by course ID

C. `Databases | 1 | 94.0`, `Algorithms | 1 | 89.5`, `Genetics | 1 | 88.0`

D. `Databases | 2 | 188.0`, `Algorithms | 2 | 179.0`, `Genetics | 1 | 88.0`

### Question 3 — Challenge: LEFT JOIN and COUNT

```sql
SELECT s.name, COUNT(e.course_id) AS course_count
FROM students AS s
LEFT JOIN enrollments AS e ON s.student_id = e.student_id
GROUP BY s.student_id, s.name
ORDER BY s.student_id;
```

What is returned?

A. `Dana | 2`, `Amir | 1`, `Noa | 2`, `Lior | 0`

B. `Dana | 2`, `Amir | 1`, `Noa | 2`; Lior is omitted because he has no enrollments

C. `Dana | 2`, `Amir | 1`, `Noa | 2`, `Lior | 1`

D. `Dana | 5`, `Amir | 5`, `Noa | 5`, `Lior | 5`

### Question 4 — Challenge: HAVING versus WHERE

```sql
SELECT course_id, AVG(grade) AS avg_grade
FROM enrollments
GROUP BY course_id
HAVING AVG(grade) > 90;
```

Which course IDs are returned?

A. `101` and `102`

B. `102` only

C. `101` only

D. `101`, `102`, and `103`




| Requirement | BSc Computer Science | MSc Computer Science | PhD |
|---|---|---|---|
| Total coursework credits | 120 | 36 | 12 |
| Required courses | Algorithms, Databases, Operating Systems | Research Methods | — |
| Elective credits | 24 | — | — |
| Final project | Required (6 credits) | — | — |
| Supervisor | — | Dr. Colonel Mustard | Prof. Plum |
| Study track | — | Thesis | — |
| Thesis topic | — | Graph Neural Networks for Social Networks | Efficient Algorithms for Large-Scale Graph Processing |
| Thesis credits | — | 12 | — |
| Minimum publications | — | — | 2 |
| Research milestones | — | — | Research proposal, Progress review, Thesis submission |


### Question 5 — MongoDB: filtering nested fields and projection

```python
list(degree_collection.find(
    {"research.minimum_publications": {"$gte": 2}},
    {"_id": 0, "degree": 1}
))
```

Assuming the three degree documents from the lecture are in the collection, what is returned?

A. The PhD document's `degree` field only.

B. The PhD and Master's documents' `degree` fields only.

C. All three documents, each with only its `degree` field.

D. The PhD document with all its fields except `_id`.

### Question 6 — Challenge: MongoDB MapReduce and counting documents

```javascript
db.degree_programs.mapReduce(
    function () {
        emit(this.level, 1);
    },
    function (key, values) {
        return Array.sum(values);
    },
    { out: { inline: 1 } }
)
```

What does this MapReduce operation calculate?

A. The total coursework credits for each degree level.

B. The number of degree-program documents for each degree level.

C. The number of required courses for each degree level.

D. The number of distinct degree levels in the collection.

### Question 7 — Challenge: MongoDB MapReduce and nested fields

```javascript
db.degree_programs.mapReduce(
    function () {
        emit(this.level, this.coursework.total_credits);
    },
    function (key, values) {
        return Array.sum(values);
    },
    { out: { inline: 1 } }
)
```

Assuming the three degree documents from the lecture are in the collection, what values are produced?

A. `Bachelor: 120`, `Master: 36`, `PhD: 12`

B. `Bachelor: 1`, `Master: 1`, `PhD: 1`

C. `Bachelor: 120`, `Master: 48`, `PhD: 12`

D. `Bachelor: 120`, `Master: 36`, `PhD: 24`


### Answer key

1. **A** — the join produces one row per enrollment; the filter keeps grades at least 90, and descending order puts 97, 95, then 91.
2. **A** — Databases: `(91 + 97) / 2 = 94`; Algorithms: `(95 + 84) / 2 = 89.5`; Genetics: `88`.
3. **A** — `LEFT JOIN` keeps students without enrollments. `COUNT(e.course_id)` ignores the `NULL` created for Lior, so his count is 0.
4. **B** — only course 102 has an average above 90: `(91 + 97) / 2 = 94`.
5. **A** — the nested-field condition matches the PhD document. The projection includes `degree` and excludes `_id`.
6. **B** — the mapper emits `(level, 1)` for every document, and the reducer sums the ones for each level. There are two PhD documents and one document at each of the other levels.
7. **A** — the mapper emits each document's degree level and its nested coursework-credit value. The reducer sums values with the same level; each level appears once in this collection.

**Challenge concepts introduced:** `LEFT JOIN`, `COUNT(column)` with `NULL`, `HAVING`, negative list indexing, dictionary unpacking, MongoDB nested-field queries, and MongoDB MapReduce (`mapReduce`, `emit`, and `Array.sum`).
